# Detecção de armas e objetos perigosos — YOLOv8n

Notebook completo para Google Colab.

Este notebook foi refeito do zero para o dataset **Dangerous Items** e **não depende de um `data.yaml` fornecido pelo dataset**.

Fluxo:

1. instala as dependências;
2. baixa e extrai o dataset;
3. identifica imagens, rótulos e divisões de treino/validação/teste;
4. reorganiza o dataset no padrão esperado pelo Ultralytics;
5. cria o `data.yaml`;
6. valida os rótulos e mostra exemplos;
7. treina o YOLOv8n;
8. avalia no teste;
9. exporta `best.pt`, ONNX e TensorFlow Lite;
10. gera um ZIP com os resultados.

Classes esperadas no dataset:

- baseball_bat
- gun
- knife
- machete
- rifle

> Antes de treinar, confira as imagens anotadas mostradas pelo notebook. Essa etapa confirma visualmente se os IDs das classes foram associados aos nomes corretos.

## 1. Instalar dependências

In [ ]:
%pip install -q "ultralytics==8.4.62" pyyaml pandas matplotlib pillow

## 2. Importações e configurações

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import shutil
import subprocess
import zipfile

from collections import Counter, defaultdict
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml

from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO


# ============================================================
# PROJETO
# ============================================================

PASTA_PROJETO = Path("/content/deteccao-armas-v2")
PASTA_DOWNLOAD = PASTA_PROJETO / "download"
PASTA_EXTRAIDA = PASTA_PROJETO / "dataset_original"
PASTA_YOLO = PASTA_PROJETO / "dataset_yolo"
PASTA_RESULTADOS = PASTA_PROJETO / "resultados"
PASTA_ARTEFATOS = PASTA_PROJETO / "artefatos"

ARQUIVO_ZIP = PASTA_DOWNLOAD / "Dangerous_Items.zip"

URL_DATASET = (
    "https://zenodo.org/records/16422779/files/"
    "Dangerous%20Items.zip?download=1"
)


# ============================================================
# DATASET
# ============================================================

EXTENSOES_IMAGEM = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

# Ordem usada apenas se o notebook não conseguir inferir a
# associação entre IDs e nomes a partir das próprias pastas.
NOMES_CLASSES_PADRAO = [
    "baseball_bat",
    "gun",
    "knife",
    "machete",
    "rifle",
]

ALIASES_CLASSES = {
    "baseball_bat": {
        "baseball_bat",
        "baseballbat",
        "baseball bat",
        "bat",
    },
    "gun": {
        "gun",
        "guns",
    },
    "knife": {
        "knife",
        "knives",
    },
    "machete": {
        "machete",
        "machetes",
    },
    "rifle": {
        "rifle",
        "rifles",
    },
}


# ============================================================
# TREINAMENTO
# ============================================================

MODELO_BASE = "yolov8n.pt"
NOME_TREINO = "yolov8n_dangerous_items"

EPOCAS = 60
TAMANHO_IMAGEM = 640
BATCH = 16
PACIENCIA = 15
SEMENTE = 42

random.seed(SEMENTE)

for pasta in [
    PASTA_DOWNLOAD,
    PASTA_EXTRAIDA,
    PASTA_YOLO,
    PASTA_RESULTADOS,
    PASTA_ARTEFATOS,
]:
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Projeto:", PASTA_PROJETO)
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Baixar o dataset

A célula usa o arquivo público `Dangerous Items.zip`.

Caso o ZIP já exista, o download não será repetido.

In [ ]:
if not ARQUIVO_ZIP.exists():
    print("Baixando dataset...")

    comando = [
        "wget",
        "-c",
        "-O",
        str(ARQUIVO_ZIP),
        URL_DATASET,
    ]

    subprocess.run(
        comando,
        check=True,
    )
else:
    print(
        "ZIP já encontrado:",
        ARQUIVO_ZIP,
    )

tamanho_gb = (
    ARQUIVO_ZIP.stat().st_size
    / 1024 ** 3
)

print(
    f"Tamanho do ZIP: {tamanho_gb:.2f} GB"
)

## 4. Extrair o dataset

In [ ]:
marcador_extracao = (
    PASTA_EXTRAIDA
    / ".extracao_concluida"
)

if not marcador_extracao.exists():
    print("Extraindo dataset...")

    with zipfile.ZipFile(
        ARQUIVO_ZIP,
        "r",
    ) as arquivo:
        arquivo.extractall(
            PASTA_EXTRAIDA
        )

    marcador_extracao.touch()
    print("Extração concluída.")
else:
    print(
        "Dataset já extraído:",
        PASTA_EXTRAIDA,
    )

## 5. Inspecionar a estrutura recebida

In [ ]:
def mostrar_estrutura(
    raiz: Path,
    profundidade_maxima: int = 4,
    limite: int = 200,
) -> None:
    exibidos = 0

    for caminho in sorted(
        raiz.rglob("*")
    ):
        profundidade = (
            len(caminho.relative_to(raiz).parts)
        )

        if profundidade > profundidade_maxima:
            continue

        prefixo = "  " * (profundidade - 1)
        tipo = "[D]" if caminho.is_dir() else "[F]"

        print(
            f"{prefixo}{tipo} {caminho.name}"
        )

        exibidos += 1

        if exibidos >= limite:
            print(
                f"... limite de {limite} itens atingido."
            )
            break


mostrar_estrutura(
    PASTA_EXTRAIDA
)

## 6. Localizar imagens, rótulos e splits

Esta etapa não procura `data.yaml`.

Ela trabalha diretamente com:

- arquivos de imagem;
- arquivos `.txt` no formato YOLO;
- nomes de pastas como `train`, `validation`, `val` e `test`.

Imagens sem arquivo de rótulo são mantidas como imagens de fundo e recebem um `.txt` vazio.

In [ ]:
PALAVRAS_SPLIT = {
    "train": {
        "train",
        "training",
        "treino",
    },
    "val": {
        "val",
        "valid",
        "validation",
        "validacao",
        "validation_set",
    },
    "test": {
        "test",
        "testing",
        "teste",
    },
}


def normalizar_texto(
    valor: str,
) -> str:
    return (
        str(valor)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


def identificar_split(
    caminho: Path,
) -> str | None:
    partes = {
        normalizar_texto(parte)
        for parte in caminho.parts
    }

    for split, aliases in (
        PALAVRAS_SPLIT.items()
    ):
        aliases_normalizados = {
            normalizar_texto(alias)
            for alias in aliases
        }

        if partes & aliases_normalizados:
            return split

    return None


def linha_yolo_valida(
    linha: str,
) -> bool:
    partes = linha.strip().split()

    if len(partes) != 5:
        return False

    try:
        classe = int(float(partes[0]))
        valores = [
            float(valor)
            for valor in partes[1:]
        ]
    except ValueError:
        return False

    return (
        classe >= 0
        and all(
            0 <= valor <= 1
            for valor in valores
        )
    )


def arquivo_rotulo_valido(
    caminho: Path,
) -> bool:
    try:
        linhas = [
            linha.strip()
            for linha in caminho.read_text(
                encoding="utf-8",
                errors="ignore",
            ).splitlines()
            if linha.strip()
        ]
    except OSError:
        return False

    # Arquivo vazio é um rótulo válido para imagem de fundo.
    if not linhas:
        return True

    return all(
        linha_yolo_valida(linha)
        for linha in linhas
    )


def ids_do_rotulo(
    caminho: Path | None,
) -> list[int]:
    if caminho is None or not caminho.exists():
        return []

    ids = []

    for linha in caminho.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        linha = linha.strip()

        if not linha:
            continue

        if not linha_yolo_valida(linha):
            continue

        ids.append(
            int(float(linha.split()[0]))
        )

    return ids


imagens = sorted(
    caminho
    for caminho in PASTA_EXTRAIDA.rglob("*")
    if (
        caminho.is_file()
        and caminho.suffix.lower()
        in EXTENSOES_IMAGEM
    )
)

arquivos_txt = sorted(
    caminho
    for caminho in PASTA_EXTRAIDA.rglob("*.txt")
    if caminho.is_file()
)

rotulos_validos = [
    caminho
    for caminho in arquivos_txt
    if arquivo_rotulo_valido(caminho)
]

indice_rotulos_por_nome = defaultdict(list)

for rotulo in rotulos_validos:
    indice_rotulos_por_nome[
        rotulo.stem
    ].append(rotulo)


def localizar_rotulo(
    imagem: Path,
) -> Path | None:
    candidatos = [
        imagem.with_suffix(".txt"),
        imagem.parent.parent
        / "labels"
        / f"{imagem.stem}.txt",
    ]

    partes = list(imagem.parts)

    for indice, parte in enumerate(partes):
        if normalizar_texto(parte) == "images":
            partes_rotulo = partes.copy()
            partes_rotulo[indice] = "labels"

            candidatos.append(
                Path(*partes_rotulo)
                .with_suffix(".txt")
            )

    for candidato in candidatos:
        if (
            candidato.exists()
            and arquivo_rotulo_valido(candidato)
        ):
            return candidato

    candidatos_nome = (
        indice_rotulos_por_nome.get(
            imagem.stem,
            [],
        )
    )

    if not candidatos_nome:
        return None

    split_imagem = identificar_split(
        imagem
    )

    candidatos_mesmo_split = [
        caminho
        for caminho in candidatos_nome
        if identificar_split(caminho)
        == split_imagem
    ]

    if candidatos_mesmo_split:
        return min(
            candidatos_mesmo_split,
            key=lambda caminho: len(
                caminho.parts
            ),
        )

    if len(candidatos_nome) == 1:
        return candidatos_nome[0]

    return min(
        candidatos_nome,
        key=lambda caminho: len(
            caminho.parts
        ),
    )


registros = []

for imagem in imagens:
    registros.append(
        {
            "imagem_original": imagem,
            "rotulo_original": localizar_rotulo(
                imagem
            ),
            "split": identificar_split(
                imagem
            ),
        }
    )

print("Imagens encontradas:", len(imagens))
print("TXT encontrados:", len(arquivos_txt))
print("Rótulos YOLO válidos:", len(rotulos_validos))

print(
    "Splits detectados:",
    Counter(
        registro["split"]
        for registro in registros
    ),
)

print(
    "Imagens sem rótulo:",
    sum(
        registro["rotulo_original"] is None
        for registro in registros
    ),
)

## 7. Garantir os splits

O dataset publicado já possui treino, validação e teste.

Caso os nomes das pastas não permitam identificá-los, o notebook aplica uma divisão reprodutível de 70% / 15% / 15%.

In [ ]:
splits_detectados = {
    registro["split"]
    for registro in registros
    if registro["split"] is not None
}

if not {"train", "val", "test"} <= splits_detectados:
    print(
        "Não foi possível reconhecer os três splits pelas pastas."
    )

    print(
        "Aplicando divisão aleatória 70/15/15."
    )

    random.Random(SEMENTE).shuffle(
        registros
    )

    total = len(registros)
    fim_treino = int(total * 0.70)
    fim_validacao = int(total * 0.85)

    for indice, registro in enumerate(
        registros
    ):
        if indice < fim_treino:
            registro["split"] = "train"
        elif indice < fim_validacao:
            registro["split"] = "val"
        else:
            registro["split"] = "test"

contagem_splits = Counter(
    registro["split"]
    for registro in registros
)

print(
    "Divisão final:",
    contagem_splits,
)

for split in ["train", "val", "test"]:
    if contagem_splits[split] == 0:
        raise RuntimeError(
            f"O split '{split}' ficou vazio."
        )

## 8. Descobrir a ordem das classes

O notebook tenta inferir a associação entre o ID numérico do rótulo e o nome da classe usando os nomes das pastas.

Quando isso não for possível, será usada a ordem documentada no notebook:

`0: baseball_bat, 1: gun, 2: knife, 3: machete, 4: rifle`

In [ ]:
def classe_no_caminho(
    caminho: Path,
) -> str | None:
    partes = {
        normalizar_texto(parte)
        for parte in caminho.parts
    }

    for classe, aliases in (
        ALIASES_CLASSES.items()
    ):
        aliases_normalizados = {
            normalizar_texto(alias)
            for alias in aliases
        }

        if partes & aliases_normalizados:
            return classe

    return None


votos_ids = defaultdict(Counter)

for registro in registros:
    classe_pasta = classe_no_caminho(
        registro["imagem_original"]
    )

    if classe_pasta is None:
        continue

    ids = ids_do_rotulo(
        registro["rotulo_original"]
    )

    if len(set(ids)) == 1:
        votos_ids[classe_pasta][
            ids[0]
        ] += 1

mapeamento_inferido = {}

for classe, votos in votos_ids.items():
    if votos:
        id_mais_comum, quantidade = (
            votos.most_common(1)[0]
        )

        mapeamento_inferido[
            id_mais_comum
        ] = classe

        print(
            f"Inferido: ID {id_mais_comum} = "
            f"{classe} ({quantidade} votos)"
        )

ids_existentes = sorted(
    {
        id_classe
        for registro in registros
        for id_classe in ids_do_rotulo(
            registro["rotulo_original"]
        )
    }
)

print(
    "IDs encontrados nos rótulos:",
    ids_existentes,
)

if (
    len(mapeamento_inferido) == 5
    and set(mapeamento_inferido) == set(range(5))
):
    classes_dataset = {
        indice: mapeamento_inferido[indice]
        for indice in range(5)
    }

    print(
        "Mapeamento inferido pelas pastas."
    )
else:
    classes_dataset = {
        indice: nome
        for indice, nome in enumerate(
            NOMES_CLASSES_PADRAO
        )
    }

    print(
        "Não foi possível inferir todo o mapeamento."
    )

    print(
        "Usando a ordem padrão:",
        classes_dataset,
    )

if ids_existentes:
    maior_id = max(ids_existentes)

    if maior_id >= len(classes_dataset):
        raise RuntimeError(
            "Existe um ID de classe maior que a "
            "quantidade de nomes configurados. "
            f"Maior ID: {maior_id}. "
            f"Classes: {classes_dataset}"
        )

print(
    "Classes finais:",
    classes_dataset,
)

## 9. Reorganizar no padrão Ultralytics

Estrutura criada:

```text
dataset_yolo/
├── train/
│   ├── images/
│   └── labels/
├── val/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

O notebook usa links físicos quando possível, evitando duplicar o espaço ocupado pelas imagens.

In [ ]:
if PASTA_YOLO.exists():
    shutil.rmtree(
        PASTA_YOLO
    )

for split in [
    "train",
    "val",
    "test",
]:
    (
        PASTA_YOLO
        / split
        / "images"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    (
        PASTA_YOLO
        / split
        / "labels"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )


def copiar_ou_linkar(
    origem: Path,
    destino: Path,
) -> None:
    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    try:
        os.link(
            origem,
            destino,
        )
    except OSError:
        shutil.copy2(
            origem,
            destino,
        )


manifesto = []

for indice, registro in enumerate(
    registros
):
    imagem_original = (
        registro["imagem_original"]
    )

    rotulo_original = (
        registro["rotulo_original"]
    )

    split = registro["split"]

    relativo = str(
        imagem_original.relative_to(
            PASTA_EXTRAIDA
        )
    )

    hash_curto = hashlib.sha1(
        relativo.encode("utf-8")
    ).hexdigest()[:10]

    nome_base = (
        f"{indice:06d}_"
        f"{hash_curto}_"
        f"{imagem_original.stem}"
    )

    destino_imagem = (
        PASTA_YOLO
        / split
        / "images"
        / f"{nome_base}"
        f"{imagem_original.suffix.lower()}"
    )

    destino_rotulo = (
        PASTA_YOLO
        / split
        / "labels"
        / f"{nome_base}.txt"
    )

    copiar_ou_linkar(
        imagem_original,
        destino_imagem,
    )

    if (
        rotulo_original is not None
        and rotulo_original.exists()
    ):
        copiar_ou_linkar(
            rotulo_original,
            destino_rotulo,
        )
    else:
        destino_rotulo.write_text(
            "",
            encoding="utf-8",
        )

    manifesto.append(
        {
            "split": split,
            "imagem_original": str(
                imagem_original
            ),
            "rotulo_original": (
                str(rotulo_original)
                if rotulo_original is not None
                else ""
            ),
            "imagem_yolo": str(
                destino_imagem
            ),
            "rotulo_yolo": str(
                destino_rotulo
            ),
            "imagem_fundo": (
                rotulo_original is None
            ),
        }
    )

df_manifesto = pd.DataFrame(
    manifesto
)

caminho_manifesto = (
    PASTA_PROJETO
    / "manifesto_dataset.csv"
)

df_manifesto.to_csv(
    caminho_manifesto,
    index=False,
)

print(
    "Dataset reorganizado:",
    PASTA_YOLO,
)

print(
    df_manifesto.groupby(
        "split"
    ).size()
)

print(
    "Manifesto:",
    caminho_manifesto,
)

## 10. Criar o `data.yaml`

In [ ]:
CAMINHO_YAML = (
    PASTA_YOLO
    / "data.yaml"
)

dados_yaml = {
    "path": str(
        PASTA_YOLO
    ),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": classes_dataset,
}

CAMINHO_YAML.write_text(
    yaml.safe_dump(
        dados_yaml,
        allow_unicode=True,
        sort_keys=False,
    ),
    encoding="utf-8",
)

print(
    CAMINHO_YAML.read_text(
        encoding="utf-8"
    )
)

## 11. Validar o dataset preparado

Esta etapa verifica:

- quantidade de imagens e rótulos;
- linhas inválidas;
- IDs fora do intervalo;
- quantidade de objetos por classe;
- imagens de fundo.

In [ ]:
def validar_dataset_yolo(
    raiz: Path,
    classes: dict[int, str],
) -> pd.DataFrame:
    linhas_resumo = []
    erros = []
    objetos_por_classe = Counter()

    for split in [
        "train",
        "val",
        "test",
    ]:
        pasta_imagens = (
            raiz
            / split
            / "images"
        )

        pasta_rotulos = (
            raiz
            / split
            / "labels"
        )

        imagens_split = sorted(
            caminho
            for caminho in pasta_imagens.iterdir()
            if caminho.suffix.lower()
            in EXTENSOES_IMAGEM
        )

        fundos = 0
        total_objetos = 0

        for imagem in imagens_split:
            rotulo = (
                pasta_rotulos
                / f"{imagem.stem}.txt"
            )

            if not rotulo.exists():
                erros.append(
                    f"Rótulo ausente: {imagem}"
                )
                continue

            linhas = [
                linha.strip()
                for linha in rotulo.read_text(
                    encoding="utf-8",
                    errors="ignore",
                ).splitlines()
                if linha.strip()
            ]

            if not linhas:
                fundos += 1
                continue

            for numero_linha, linha in enumerate(
                linhas,
                start=1,
            ):
                if not linha_yolo_valida(
                    linha
                ):
                    erros.append(
                        f"Inválido: {rotulo}, "
                        f"linha {numero_linha}: {linha}"
                    )
                    continue

                id_classe = int(
                    float(
                        linha.split()[0]
                    )
                )

                if id_classe not in classes:
                    erros.append(
                        f"ID fora do intervalo: "
                        f"{rotulo}: {id_classe}"
                    )
                    continue

                objetos_por_classe[
                    id_classe
                ] += 1

                total_objetos += 1

        linhas_resumo.append(
            {
                "split": split,
                "imagens": len(
                    imagens_split
                ),
                "imagens_fundo": fundos,
                "objetos": total_objetos,
            }
        )

    if erros:
        print(
            "Primeiros erros:"
        )

        for erro in erros[:20]:
            print(
                " -",
                erro,
            )

        raise RuntimeError(
            f"Foram encontrados {len(erros)} "
            "problemas no dataset."
        )

    print(
        "Objetos por classe:"
    )

    for id_classe in sorted(
        classes
    ):
        print(
            f"  {id_classe} - "
            f"{classes[id_classe]}: "
            f"{objetos_por_classe[id_classe]}"
        )

    return pd.DataFrame(
        linhas_resumo
    )


resumo_dataset = (
    validar_dataset_yolo(
        PASTA_YOLO,
        classes_dataset,
    )
)

display(
    resumo_dataset
)

## 12. Conferir visualmente as anotações

Observe os nomes das classes desenhados nas caixas.

Caso as caixas estejam corretas, mas os nomes apareçam trocados, ajuste apenas `NOMES_CLASSES_PADRAO` na configuração e execute novamente a partir da etapa de criação do YAML.

In [ ]:
CORES = [
    "red",
    "lime",
    "cyan",
    "yellow",
    "magenta",
]


def desenhar_rotulos(
    caminho_imagem: Path,
    caminho_rotulo: Path,
    classes: dict[int, str],
) -> Image.Image:
    imagem = Image.open(
        caminho_imagem
    ).convert("RGB")

    largura, altura = imagem.size
    desenho = ImageDraw.Draw(
        imagem
    )

    if caminho_rotulo.exists():
        linhas = caminho_rotulo.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()

        for linha in linhas:
            linha = linha.strip()

            if not linha:
                continue

            partes = linha.split()

            if len(partes) != 5:
                continue

            id_classe = int(
                float(partes[0])
            )

            x_centro, y_centro, caixa_largura, caixa_altura = [
                float(valor)
                for valor in partes[1:]
            ]

            x1 = int(
                (
                    x_centro
                    - caixa_largura / 2
                )
                * largura
            )

            y1 = int(
                (
                    y_centro
                    - caixa_altura / 2
                )
                * altura
            )

            x2 = int(
                (
                    x_centro
                    + caixa_largura / 2
                )
                * largura
            )

            y2 = int(
                (
                    y_centro
                    + caixa_altura / 2
                )
                * altura
            )

            cor = CORES[
                id_classe
                % len(CORES)
            ]

            desenho.rectangle(
                [x1, y1, x2, y2],
                outline=cor,
                width=4,
            )

            texto = (
                f"{id_classe}: "
                f"{classes[id_classe]}"
            )

            desenho.rectangle(
                [
                    x1,
                    max(0, y1 - 22),
                    x1 + max(
                        100,
                        len(texto) * 8,
                    ),
                    y1,
                ],
                fill=cor,
            )

            desenho.text(
                (
                    x1 + 3,
                    max(0, y1 - 20),
                ),
                texto,
                fill="black",
            )

    return imagem


imagens_treino = sorted(
    (
        PASTA_YOLO
        / "train"
        / "images"
    ).iterdir()
)

amostras = random.sample(
    imagens_treino,
    k=min(
        12,
        len(imagens_treino),
    ),
)

plt.figure(
    figsize=(18, 14)
)

for indice, caminho_imagem in enumerate(
    amostras,
    start=1,
):
    caminho_rotulo = (
        PASTA_YOLO
        / "train"
        / "labels"
        / f"{caminho_imagem.stem}.txt"
    )

    imagem_anotada = desenhar_rotulos(
        caminho_imagem,
        caminho_rotulo,
        classes_dataset,
    )

    plt.subplot(
        3,
        4,
        indice,
    )

    plt.imshow(
        imagem_anotada
    )

    plt.axis(
        "off"
    )

plt.tight_layout()
plt.show()

## 13. Treinar o YOLOv8n

Ative uma GPU no Colab:

`Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU não disponível. Ative uma GPU T4 no Colab."
    )

modelo = YOLO(
    MODELO_BASE
)

resultado_treino = modelo.train(
    data=str(
        CAMINHO_YAML
    ),
    epochs=EPOCAS,
    imgsz=TAMANHO_IMAGEM,
    batch=BATCH,
    patience=PACIENCIA,
    device=0,
    workers=2,
    seed=SEMENTE,
    deterministic=True,
    project=str(
        PASTA_RESULTADOS
    ),
    name=NOME_TREINO,
    exist_ok=True,
    cache=False,
    plots=True,
    close_mosaic=10,
    verbose=True,
)

## 14. Localizar e copiar os melhores pesos

In [ ]:
PASTA_TREINO = (
    PASTA_RESULTADOS
    / NOME_TREINO
)

MELHOR_PESO = (
    PASTA_TREINO
    / "weights"
    / "best.pt"
)

ULTIMO_PESO = (
    PASTA_TREINO
    / "weights"
    / "last.pt"
)

if not MELHOR_PESO.exists():
    raise FileNotFoundError(
        f"best.pt não encontrado: {MELHOR_PESO}"
    )

shutil.copy2(
    MELHOR_PESO,
    PASTA_ARTEFATOS
    / "best.pt",
)

if ULTIMO_PESO.exists():
    shutil.copy2(
        ULTIMO_PESO,
        PASTA_ARTEFATOS
        / "last.pt",
    )

print(
    "Melhor peso:",
    MELHOR_PESO,
)

## 15. Avaliar no conjunto de teste

In [ ]:
melhor_modelo = YOLO(
    str(MELHOR_PESO)
)

metricas = melhor_modelo.val(
    data=str(
        CAMINHO_YAML
    ),
    split="test",
    imgsz=TAMANHO_IMAGEM,
    batch=BATCH,
    device=0,
    workers=2,
    plots=True,
    project=str(
        PASTA_RESULTADOS
    ),
    name="avaliacao_teste",
)

resumo_metricas = {
    "map50_95": float(
        metricas.box.map
    ),
    "map50": float(
        metricas.box.map50
    ),
    "map75": float(
        metricas.box.map75
    ),
    "map_por_classe": [
        float(valor)
        for valor in metricas.box.maps
    ],
    "classes": classes_dataset,
}

caminho_metricas = (
    PASTA_ARTEFATOS
    / "metricas_teste.json"
)

caminho_metricas.write_text(
    json.dumps(
        resumo_metricas,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        resumo_metricas,
        ensure_ascii=False,
        indent=2,
    )
)

## 16. Testar predições em imagens

In [ ]:
imagens_teste = sorted(
    (
        PASTA_YOLO
        / "test"
        / "images"
    ).iterdir()
)

fontes_teste = random.sample(
    imagens_teste,
    k=min(
        12,
        len(imagens_teste),
    ),
)

melhor_modelo.predict(
    source=[
        str(caminho)
        for caminho in fontes_teste
    ],
    imgsz=TAMANHO_IMAGEM,
    conf=0.25,
    save=True,
    project=str(
        PASTA_RESULTADOS
    ),
    name="predicoes_teste",
    exist_ok=True,
)

print(
    "Predições salvas em:",
    PASTA_RESULTADOS
    / "predicoes_teste",
)

## 17. Exportar para ONNX e TensorFlow Lite

A exportação INT8 usa imagens do próprio dataset para calibração.

In [ ]:
caminhos_exportados = {}


# ------------------------------------------------------------
# ONNX
# ------------------------------------------------------------

try:
    caminho_onnx = melhor_modelo.export(
        format="onnx",
        imgsz=TAMANHO_IMAGEM,
        dynamic=True,
        simplify=True,
        opset=12,
    )

    caminhos_exportados[
        "onnx"
    ] = str(caminho_onnx)

    print(
        "ONNX:",
        caminho_onnx,
    )

except Exception as erro:
    print(
        "Falha na exportação ONNX:",
        erro,
    )


# ------------------------------------------------------------
# TFLITE FLOAT32
# ------------------------------------------------------------

try:
    caminho_tflite = melhor_modelo.export(
        format="tflite",
        imgsz=TAMANHO_IMAGEM,
    )

    caminhos_exportados[
        "tflite_float"
    ] = str(caminho_tflite)

    print(
        "TFLite:",
        caminho_tflite,
    )

except Exception as erro:
    print(
        "Falha na exportação TFLite:",
        erro,
    )


# ------------------------------------------------------------
# TFLITE INT8
# ------------------------------------------------------------

try:
    caminho_tflite_int8 = (
        melhor_modelo.export(
            format="tflite",
            imgsz=TAMANHO_IMAGEM,
            int8=True,
            data=str(
                CAMINHO_YAML
            ),
        )
    )

    caminhos_exportados[
        "tflite_int8"
    ] = str(
        caminho_tflite_int8
    )

    print(
        "TFLite INT8:",
        caminho_tflite_int8,
    )

except Exception as erro:
    print(
        "Falha na exportação TFLite INT8:",
        erro,
    )

print(
    json.dumps(
        caminhos_exportados,
        indent=2,
        ensure_ascii=False,
    )
)

## 18. Reunir todos os artefatos

In [ ]:
def copiar_exportacao(
    caminho_texto: str,
    nome: str,
) -> None:
    caminho = Path(
        caminho_texto
    )

    if caminho.is_file():
        shutil.copy2(
            caminho,
            PASTA_ARTEFATOS
            / caminho.name,
        )

        return

    if caminho.is_dir():
        destino = (
            PASTA_ARTEFATOS
            / nome
        )

        if destino.exists():
            shutil.rmtree(
                destino
            )

        shutil.copytree(
            caminho,
            destino,
        )


for nome, caminho in (
    caminhos_exportados.items()
):
    copiar_exportacao(
        caminho,
        nome,
    )

shutil.copy2(
    CAMINHO_YAML,
    PASTA_ARTEFATOS
    / "data.yaml",
)

shutil.copy2(
    caminho_manifesto,
    PASTA_ARTEFATOS
    / "manifesto_dataset.csv",
)

arquivo_classes = (
    PASTA_ARTEFATOS
    / "classes.json"
)

arquivo_classes.write_text(
    json.dumps(
        classes_dataset,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Artefatos:"
)

for caminho in sorted(
    PASTA_ARTEFATOS.rglob("*")
):
    if caminho.is_file():
        tamanho_mb = (
            caminho.stat().st_size
            / 1024 ** 2
        )

        print(
            f"- {caminho.relative_to(PASTA_ARTEFATOS)} "
            f"({tamanho_mb:.2f} MB)"
        )

## 19. Criar ZIP para download

In [ ]:
ARQUIVO_RESULTADOS = (
    PASTA_PROJETO
    / "resultados_deteccao_armas_v2.zip"
)

if ARQUIVO_RESULTADOS.exists():
    ARQUIVO_RESULTADOS.unlink()

with zipfile.ZipFile(
    ARQUIVO_RESULTADOS,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as arquivo_zip:
    for caminho in PASTA_ARTEFATOS.rglob("*"):
        if caminho.is_file():
            arquivo_zip.write(
                caminho,
                caminho.relative_to(
                    PASTA_ARTEFATOS
                ),
            )

    for pasta_resultado in [
        PASTA_TREINO,
        PASTA_RESULTADOS
        / "avaliacao_teste",
        PASTA_RESULTADOS
        / "predicoes_teste",
    ]:
        if not pasta_resultado.exists():
            continue

        for caminho in pasta_resultado.rglob("*"):
            if caminho.is_file():
                arquivo_zip.write(
                    caminho,
                    Path("resultados")
                    / caminho.relative_to(
                        PASTA_RESULTADOS
                    ),
                )

tamanho_zip_mb = (
    ARQUIVO_RESULTADOS.stat().st_size
    / 1024 ** 2
)

print(
    "ZIP criado:",
    ARQUIVO_RESULTADOS,
)

print(
    f"Tamanho: {tamanho_zip_mb:.2f} MB"
)

## 20. Baixar o ZIP

In [ ]:
from google.colab import files

files.download(
    str(ARQUIVO_RESULTADOS)
)